# Lesson 02 - Khám phá Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** là một khung thống nhất để xây dựng các đại lý AI. Nó cung cấp một kiến trúc sạch sẽ, có thể kết hợp với bốn khối xây dựng cốt lõi:

- **Client** – kết nối với điểm cuối mô hình AI và xử lý giao tiếp
- **Agent** – bao bọc một client với hướng dẫn và định nghĩa công cụ
- **Tools** – mở rộng khả năng đại lý với các chức năng tùy chỉnh mà mô hình có thể gọi
- **Session** – duy trì lịch sử cuộc trò chuyện cho các tương tác đa lượt

Trong bài học này, chúng ta sẽ xây dựng một **đại lý đặt chuyến đi** kiểm tra tính khả dụng của điểm đến sử dụng các khái niệm này.


## Cài đặt


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Hiểu về Kiến trúc Khung Agent

Microsoft Agent Framework tuân theo một kiến trúc phân lớp:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – Một `AzureAIProjectAgentProvider` kết nối với một triển khai Azure OpenAI. Nó xử lý xác thực, định dạng yêu cầu và phân tích phản hồi.
2. **Agent** – Được tạo từ client thông qua `provider.create_agent()`, agent kết hợp truy cập mô hình với hướng dẫn (system prompt) và công cụ.
3. **Tools** – Các hàm Python được trang trí bằng `@tool` mà agent có thể gọi để thực hiện hành động hoặc lấy dữ liệu.
4. **Session** – Một đối tượng `AgentSession` (được tạo qua `agent.create_session()`) lưu trữ lịch sử hội thoại, cho phép đối thoại đa lượt trong đó agent nhớ ngữ cảnh trước đó.

Hãy xây dựng từng lớp một từng bước.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Thêm Công Cụ với Trình Trang Trí @tool

Công cụ cho phép tác nhân thực hiện các hành động vượt ra ngoài việc tạo văn bản. Trình trang trí `@tool` biến một hàm Python thông thường thành thứ mà tác nhân có thể gọi.

Các điểm chính:
- Sử dụng `Annotated[type, "mô tả"]` để mô hình hiểu từng tham số.
- Docstring trở thành mô tả công cụ mà mô hình thấy.
- `approval_mode="never_require"` nghĩa là công cụ chạy tự động mà không cần xác nhận của người dùng.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Tạo một Đại lý với Công cụ

Bây giờ chúng ta kết hợp client, hướng dẫn và công cụ thành một đại lý. `instructions` đóng vai trò như lời nhắc hệ thống — chúng xác định nhân cách và hành vi của đại lý.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Cuộc Hội Thoại Nhiều Vòng Với Các Phiên

Một `AgentSession` (được tạo qua `agent.create_session()`) theo dõi tất cả các tin nhắn trong một cuộc hội thoại. Bằng cách truyền cùng một phiên cho mỗi lần gọi `agent.run()`, agent có thể truy cập vào toàn bộ lịch sử cuộc hội thoại và có thể tham khảo lại các tin nhắn trước đó.

Chúng tôi truyền `tools=[check_destination_availability]` để agent có thể gọi trình kiểm tra khả dụng của chúng tôi trong mỗi vòng.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Tóm tắt

Trong bài học này, bạn đã khám phá bốn trụ cột của Microsoft Agent Framework:

| Khái niệm | Những gì bạn đã học |
|---------|------------------|
| **Client** | `AzureAIProjectAgentProvider` kết nối với Azure OpenAI bằng xác thực dựa trên thông tin đăng nhập |
| **Agent** | `provider.create_agent()` đóng gói một kết nối mô hình với hướng dẫn và tên |
| **Tools** | Bộ trang trí `@tool` cho phép các hàm Python để agent gọi |
| **Session** | `agent.create_session()` duy trì lịch sử cuộc trò chuyện qua nhiều lượt |

Những khối xây dựng này kết hợp với nhau để tạo ra các agent có thể giữ cuộc trò chuyện tự nhiên, gọi các hàm bên ngoài và duy trì ngữ cảnh — nền tảng cho các mẫu agent nâng cao hơn trong các bài học tiếp theo.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố từ chối trách nhiệm**:  
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc không chính xác. Tài liệu gốc bằng ngôn ngữ bản địa nên được coi là nguồn đáng tin cậy nhất. Đối với những thông tin quan trọng, khuyến nghị sử dụng dịch vụ dịch thuật chuyên nghiệp của con người. Chúng tôi không chịu trách nhiệm đối với bất kỳ sự hiểu lầm hoặc diễn giải sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
